# Tập 3.5 · Định vị trên bản đồ có sẵn: robot bị bắt cóc 🗺️

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap35/tap35_mcl.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 3 · ⏱ ~45 phút · Giáo trình: Principles of Robot Autonomy, [Chương 13](https://porabook.com/book/robot-localization/)

Monte Carlo Localization: bộ lọc hạt + bản đồ + LiDAR. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video (~30 giây).
2. **Tự viết** phép dò tia, trọng số theo bản đồ và mẹo thả hạt chống bắt cóc (3 bài); ô chấm chạy lại cả 100 bước.
3. Vẽ đám hạt trên bản đồ.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap35")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import mcl as goc                  # mô phỏng trong video (chạy 100 bước lúc import)
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
for k in (1, 4, 51, 52, 55, 60, 100):
    r = goc.LOG[k - 1]
    print(f"bước {k:3d}{' (BẮT CÓC)' if r['bat_coc'] else ''}: có thả hạt {r['tha']['gan'] * 100:5.1f}% đúng chỗ · không thả {r['khong']['gan'] * 100:5.1f}%")

## 2. Bài tập

Mỗi tia: $p = 0{,}9\,e^{-\frac12((d_{dự đoán} - z)/0{,}6)^2} + 0{,}1\cdot 0{,}2$; trọng số hạt $= \prod_{8\ tia} p$ (cộng log), hạt trong tường = 0.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `quet(X)` | dò 8 tia trên bản đồ lưới (bước 5 cm, tầm 6 m) |
| **2** | `trong_so(P, z)` | trọng số theo 8 tia và bản đồ |
| **3** | `tha_hat(P, chon, hat_moi)` | thay các hạt được chọn bằng hạt ngẫu nhiên |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np

from mcl import BUOC_TIA, GOC_TIA, K, P_DEU, SIG_TIA, TAM, chiem  # noqa: F401  (bản đồ và thông số có sẵn, trong ../sim)


def quet(X):
    """Bài 1: dò tia LiDAR cho mảng tư thế X (n, 3). Trả về mảng khoảng cách (n, 8).

    Tia thứ j của tư thế (x, y, θ) có hướng θ + GOC_TIA[j]. Đi dọc tia theo các bước d trong BUOC_TIA (5 cm, 10 cm, ..., 6 m):
    khoảng cách của tia là d ĐẦU TIÊN mà chiem(x + d·cos, y + d·sin) là True. Không chạm gì thì bằng TAM (6 m).
    Gợi ý: làm cho cả mảng cùng lúc; giữ một mảng "da" (đã chạm) để không ghi đè."""
    raise NotImplementedError("TODO Bài 1: dò tia")


def trong_so(P, z):
    """Bài 2 (Eq. 13.3): trọng số các hạt từ 8 tia đo được z.

    du = quet(P) (8 tia dự đoán cho mỗi hạt). Mỗi tia: p = (1 − P_DEU)·exp(−½((du − z)/SIG_TIA)²) + P_DEU·0.2
    Trọng số của hạt = tích của 8 p. Làm bằng log cho khỏi tràn số: logw = Σ log p; hạt nằm trong tường thì logw = −∞ (Eq. 13.2).
    w = exp(logw − max(logw)), rồi chuẩn hoá tổng = 1."""
    raise NotImplementedError("TODO Bài 2: trọng số")


def tha_hat(P, chon, hat_moi):
    """Bài 3 (ghi chú lề §13.6, chống bắt cóc): thay các hạt ở vị trí chon (mảng chỉ số) bằng hat_moi (rải ngẫu nhiên trên bản đồ).

    Trả về mảng hạt mới (đừng sửa thẳng vào P truyền vào: hãy copy trước)."""
    raise NotImplementedError("TODO Bài 3: thả hạt ngẫu nhiên")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong Bài 1, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Robot ở các chỗ khác nhau "thấy" gì? Thử quét LiDAR ở vài tư thế.

In [ ]:
import loi_giai
for tu_the in [(1.2, 2.6, 0.0), (3.0, 3.0, 1.0), (9.0, 5.0, -2.0), (2.9, 4.05, 1.57)]:
    d = loi_giai.quet(np.array([tu_the]))[0]
    print(f"tư thế {tu_the}: {' '.join(f'{v:4.2f}' for v in d)} m")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap35).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)